# Robustness of Chimera States in Coupled FitzHugh-Nagumo Oscillators

This notebook follows the order of the paper. First I set up the FHN model, then I test what happens when I change the oscillators or the connection pattern. A flat part in the phase-velocity plot means coherent motion, while an arc means incoherent motion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


In [ ]:
# the fractal connection matrix is used in the last section
def build_fractal_matrix(base_pattern, N, iterations=4):
    current_string = [1]
    for _ in range(iterations):
        next_string = []
        for symbol in current_string:
            next_string.extend(base_pattern if symbol == 1 else [0] * len(base_pattern))
        current_string = next_string

    kernel = np.array(current_string, dtype=float)
    kernel[0] = 0.0 #remove self coupling
    C = np.zeros((N, N))
    for k in range(N):
        C[k] = np.roll(kernel, k)
    return C


## II. Model and shared setup

This cell has the FHN model and the helper functions used in every simulation below. I made N smaller so it runs without taking forever. N_run = 1000 & T_run = 2500 are the paper values.

In [ ]:
N_run = 300                 
T_run = 1000              
n_times = 300
epsilon = 0.05
a_mean = 0.5
phi = np.pi / 2 - 0.1
buu, buv = np.cos(phi), np.sin(phi)
bvu, bvv = -np.sin(phi), np.cos(phi)

def ring_matrix(N, R):
    C = np.zeros((N, N))
    for k in range(N):
        C[k, (k + np.r_[np.arange(-R, 0), np.arange(1, R + 1)]) % N] = 1
    return C

def add_shortcuts(C, p, seed=1):
    #keep the original ring links and randomly add some extra ones
    rng = np.random.default_rng(seed)
    N = len(C)
    candidates = np.triu((C == 0), 1)
    new_links = candidates & (rng.random((N, N)) < p)
    return np.maximum(C, new_links + new_links.T).astype(float)

def rewire_links(C, p, seed=1):
    #instead of adding links, replace some of the old ring links
    rng = np.random.default_rng(seed)
    C_new = C.copy()
    edges = np.argwhere(np.triu(C_new, 1) == 1)
    for i, j in edges:
        if rng.random() < p:
            C_new[i, j] = C_new[j, i] = 0
            choices = np.flatnonzero((C_new[i] == 0) & (np.arange(len(C_new)) != i))
            new_j = rng.choice(choices)
            C_new[i, new_j] = C_new[new_j, i] = 1
    return C_new

def symmetric_gap_matrix(N, R1, R2, G):
    #R1 links are close to the node, then there is a gap, then R2 farther links
    C = np.zeros((N, N))
    offsets = np.r_[np.arange(-R1, R1 + 1),
                    np.arange(R1 + G + 1, R1 + G + R2 + 1),
                   -np.arange(R1 + G + 1, R1 + G + R2 + 1)]
    offsets = offsets[offsets != 0]
    for k in range(N):
        C[k, (k + offsets) % N] = 1
    return C

def one_sided_gap_matrix(N, R1, R2, G):
    #all links point to the right here, with one gap in between
    C = np.zeros((N, N))
    offsets = np.r_[np.arange(1, R1 + 1), np.arange(R1 + G + 1, R1 + G + R2 + 1)]
    for k in range(N):
        C[k, (k + offsets) % N] = 1
    return C

def simulate_fhn(C, sigma, a_values=a_mean, seed=42):
    N = C.shape[0]
    degree = C.sum(axis=1)
    if np.any(degree == 0):
        raise ValueError('Every oscillator needs at least one connection.')
    a_values = np.full(N, a_values) if np.isscalar(a_values) else np.asarray(a_values)
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2*np.pi, N)
    y0 = np.r_[2*np.cos(theta), 2*np.sin(theta)]
    t_eval = np.linspace(0, T_run, n_times)

    def fhn_deriv(t, y):
        u, v = y[:N], y[N:]
        diff_u = C @ u - degree*u
        diff_v = C @ v - degree*v
        coupling_u = buu*diff_u + buv*diff_v
        coupling_v = bvu*diff_u + bvv*diff_v
        du = (u - u**3/3 - v + sigma*coupling_u/degree) / epsilon
        dv = u + a_values + sigma*coupling_v/degree
        return np.r_[du, dv]

    sol = solve_ivp(fhn_deriv, (0, T_run), y0, method='DOP853', t_eval=t_eval, rtol=1e-5, atol=1e-7)
    if not sol.success:
        raise RuntimeError(sol.message)
    burn = len(t_eval) // 2
    phase = np.unwrap(np.arctan2(sol.y[N:, burn:], sol.y[:N, burn:]), axis=1)
    omega = (phase[:, -1] - phase[:, 0]) / (t_eval[-1] - t_eval[burn])
    return sol.y[:N, -1], np.roll(omega, N//2)

def plot_case(u, omega, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(u, 'k.', ms=2)
    ax[0].set(xlabel='Node k', ylabel='u_k', title='Snapshot')
    ax[1].plot(omega, 'k.', ms=2)
    ax[1].set(xlabel='Node k', ylabel='Mean phase velocity', title=title)
    for axis in ax: axis.grid(alpha=0.3)
    plt.tight_layout()

def coherence_measures(omega, tolerance=0.05):
    omega_coh = np.median(omega)
    delta = np.abs(omega - omega_coh)
    return {'omega_coh': omega_coh, 'delta_omega': delta.max(),
            'N_incoh': np.mean(delta > tolerance), 'M_incoh': delta.sum()}


## III. Different oscillators

The first section was already doing this. These are the three cases in the paper: one, two, and three incoherent regions.

In [ ]:
inhomogeneous_cases = [(0.35, 0.10, 'one incoherent domain'),
                       (0.33, 0.28, 'two incoherent domains'),
                       (0.25, 0.25, 'three incoherent domains')]
for r_case, sigma_case, label in inhomogeneous_cases:
    R_case = int(r_case * N_run)
    C = ring_matrix(N_run, R_case)
    for delta_a in [0.0001, 0.001, 0.01]:
        rng = np.random.default_rng(10)
        a_k = rng.normal(a_mean, np.sqrt(delta_a), N_run)
        u, omega = simulate_fhn(C, sigma_case, a_k, seed=42)
        plot_case(u, omega, f'{label}: variance = {delta_a}')


## IV. Random extra links and rewiring

first I keep the normal ring and add random long distance links. Then I try rewiring, where some of the original links are removed and changed.

In [ ]:
R_case = int(0.35 * N_run)
C_ring = ring_matrix(N_run, R_case)
for p in [0.05, 0.10, 0.20, 0.30]:
    C_shortcuts = add_shortcuts(C_ring, p, seed=7)
    u, omega = simulate_fhn(C_shortcuts, sigma=0.10, seed=42)
    plot_case(u, omega, f'Additional random links: p = {p}')

for p in [0.05, 0.20]:
    C_rewired = rewire_links(C_ring, p, seed=7)
    u, omega = simulate_fhn(C_rewired, sigma=0.10, seed=42)
    plot_case(u, omega, f'Rewired links: p = {p}')


In [ ]:
#test like Fig. 4. It counts a profile if it has a flat part and an arc part

def looks_like_chimera(omega, tolerance=0.03):
    deviation = np.abs(omega - np.median(omega))
    coherent_fraction = np.mean(deviation < tolerance)
    return 0.15 < coherent_fraction < 0.85 and deviation.max() > 2*tolerance

RUN_STABILITY_SCAN = False    #500 runs is what the paper uses but it takes a long time
if RUN_STABILITY_SCAN:
    trials = 500
    r_values, sigma_values, p_values = [0.25, 0.30, 0.35], [0.10, 0.20, 0.30], [0, 0.05, 0.10, 0.20]
    fraction = np.zeros((len(p_values), len(r_values), len(sigma_values)))
    for ip, p in enumerate(p_values):
        for ir, r_case in enumerate(r_values):
            C0 = ring_matrix(N_run, int(r_case*N_run))
            for js, sigma_case in enumerate(sigma_values):
                hits = 0
                for trial in range(trials):
                    C = add_shortcuts(C0, p, seed=trial)
                    _, omega = simulate_fhn(C, sigma_case, seed=trial)
                    hits += looks_like_chimera(omega)
                fraction[ip, ir, js] = hits / trials
    #one heatmap for each probability so we can see how the stable area moves
    fig, axes = plt.subplots(1, len(p_values), figsize=(14, 3), sharey=True)
    for ax, p, values in zip(axes, p_values, fraction):
        im = ax.imshow(values.T, origin='lower', vmin=0, vmax=1, aspect='auto')
        ax.set(title=f'p = {p}', xlabel='coupling radius index', ylabel='coupling strength index')
    fig.colorbar(im, ax=axes, label='fraction of chimera realizations')
    plt.show()


## V. Putting gaps in the connections

Here every oscillator still has the same number of links. I only move the gap around to see how the position changes the chimera.

In [ ]:
#two equal gaps, one on each side of every node
R_total = int(0.25 * N_run)
G = int(0.10 * N_run)
gap_results = []
for R1 in [R_total, int(0.8*R_total), R_total//2, int(0.2*R_total), 0]:
    C_gap = symmetric_gap_matrix(N_run, R1, R_total - R1, G)
    u, omega = simulate_fhn(C_gap, sigma=0.30, seed=42)
    plot_case(u, omega, f'Symmetric gaps: R1 = {R1}, R2 = {R_total - R1}')
    gap_results.append((R1, coherence_measures(omega)))

for R1, measures in gap_results:
    print(f'R1 = {R1}:', measures)

#one sided case: all the links go right and there is one gap
R_one_sided = int(0.50 * N_run)
for R1 in [int(0.8*R_one_sided), R_one_sided//2, max(1, int(0.02*R_one_sided))]:
    C_gap = one_sided_gap_matrix(N_run, R1, R_one_sided - R1, G)
    u, omega = simulate_fhn(C_gap, sigma=0.30, seed=42)
    plot_case(u, omega, f'One-sided gap: R1 = {R1}, R2 = {R_one_sided - R1}')


## VI. Fractal / hierarchical gaps

Each pattern is divided by its own number of links because they do not all have the same amount. The last two patterns have the same fractal dimension but the links are arranged differently.

In [ ]:
RUN_PAPER_SCALE = False       #set True for the paper size N = 6**4 = 1296
hierarchical_iterations = 4 if RUN_PAPER_SCALE else 3
N_hierarchical = 6**hierarchical_iterations
hierarchical_patterns = [[1, 0, 0, 1, 0, 1], [1, 0, 1, 1, 1, 0],
                         [1, 1, 0, 1, 1, 1], [1, 1, 0, 0, 1, 1],
                         [1, 1, 1, 0, 1, 0]]
#the faster version uses three repeats, but the same idea
for pattern in hierarchical_patterns:
    C = build_fractal_matrix(pattern, N_hierarchical, iterations=hierarchical_iterations)
    u, omega = simulate_fhn(C, sigma=0.30, seed=42)
    d_f = np.log(sum(pattern)) / np.log(len(pattern))
    plot_case(u, omega, f'Hierarchical base {pattern}, d_f = {d_f:.4f}')
